<a href="https://colab.research.google.com/github/pGovm/creep_test_final_project/blob/main/Final_Project_Data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, datasets
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
import matplotlib.pyplot as plt

import datetime
import os
import tensorflow as tf
import time
from pathlib import Path

from tqdm import tqdm

In [2]:
!pip install thop

In [3]:
!pip install torchinfo

In [4]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# **DataSet Loader**

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
#downloads dataset from google drive
data_folder = Path('/content/drive/MyDrive/01_primary_data')

In [8]:
all_dfs = []

for csv_file in data_folder.glob("*.csv"):
    print(f"Processing: {csv_file.name}")


    df = pd.read_csv(csv_file, sep=";", skiprows=[1])

    # Convert measurement columns to numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Add experiment identifier
    df["Experiment_ID"] = csv_file.stem

    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

print("Combined DataFrame head:")
display(combined_df.head())

print("Combined DataFrame info:")
combined_df.info()

print("\nMissing values:")
print(combined_df.isna().sum())

print("\nNumber of experiments:")
print(combined_df["Experiment_ID"].nunique())

Processing: 16120245UVMS.csv
Processing: 1511007DXMS.csv
Processing: 1511011DXMS.csv
Processing: 1511013UVMS.csv
Processing: 16120250DXMS.csv
Processing: 1511010DXMS.csv
Processing: 16120248DXMS.csv
Processing: 16120251DXMS.csv
Processing: 16120252DXMS.csv
Processing: 1511012UVMS.csv
Processing: 1511009DXMS.csv
Processing: 1511008UVMS.csv
Processing: 1511009UVMS.csv
Processing: 1511008DXMS.csv
Processing: 1511007UVMS.csv
Processing: 16120244UVMS.csv
Processing: 17010248UVMS.csv
Processing: 16120253DXMS.csv
Processing: 17020264UVMS.csv
Processing: 17010253UVMS.csv
Combined DataFrame head:


,Duration,Force,Elongation,Temperature,Experiment_ID
0,0.000000,1.3,0.0,21.7,16120245UVMS
1,0.000282,1.3,0.0,21.7,16120245UVMS
2,0.000559,1.3,0.0,21.7,16120245UVMS
3,0.001595,1.3,0.0,21.9,16120245UVMS
4,0.002154,1.3,0.0,21.9,16120245UVMS


Combined DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159616 entries, 0 to 159615
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Duration       159616 non-null  float64
 1   Force          159616 non-null  float64
 2   Elongation     159616 non-null  float64
 3   Temperature    159616 non-null  float64
 4   Experiment_ID  159616 non-null  object 
dtypes: float64(4), object(1)
memory usage: 6.1+ MB

Missing values:
Duration         0
Force            0
Elongation       0
Temperature      0
Experiment_ID    0
dtype: int64

Number of experiments:
20


In [9]:
print("Missing values before cleaning:")
print(combined_df.isna().sum())

# Remove rows missing any required measurement
combined_df = combined_df.dropna(
    subset=["Duration", "Force", "Elongation", "Temperature"]
).reset_index(drop=True)

print("\nMissing values after cleaning:")
print(combined_df.isna().sum())

Missing values before cleaning:
Duration         0
Force            0
Elongation       0
Temperature      0
Experiment_ID    0
dtype: int64

Missing values after cleaning:
Duration         0
Force            0
Elongation       0
Temperature      0
Experiment_ID    0
dtype: int64


# **Data Split**

In [11]:
# Preserve chronological ordering before splitting
combined_df = (
    combined_df
    .sort_values(["Experiment_ID", "Duration"])
    .reset_index(drop=True)
)

# Normalize the features



In [16]:
# Define features (X) and potential target (y)
X = combined_df.drop('Experiment_ID', axis=1) # Features to normalize
experiment_ids = combined_df['Experiment_ID']

# Get unique Experiment_IDs to ensure data from the same experiment stays in the same set
unique_experiments = experiment_ids.unique()

# Split experiments into training, validation, and test sets (70-15-15 split)
train_exp, temp_exp = train_test_split(unique_experiments, test_size=0.3, random_state=seed)
val_exp, test_exp = train_test_split(temp_exp, test_size=0.5, random_state=seed)

print(f"Training experiments: {len(train_exp)}")
print(f"Validation experiments: {len(val_exp)}")
print(f"Test experiments: {len(test_exp)}")

# Create the actual datasets based on the experiment splits
X_train = combined_df[combined_df['Experiment_ID'].isin(train_exp)].drop('Experiment_ID', axis=1)
X_val = combined_df[combined_df['Experiment_ID'].isin(val_exp)].drop('Experiment_ID', axis=1)
X_test = combined_df[combined_df['Experiment_ID'].isin(test_exp)].drop('Experiment_ID', axis=1)

# Store Experiment_IDs for each split if needed later
experiment_ids_train = combined_df[combined_df['Experiment_ID'].isin(train_exp)]['Experiment_ID']
experiment_ids_val = combined_df[combined_df['Experiment_ID'].isin(val_exp)]['Experiment_ID']
experiment_ids_test = combined_df[combined_df['Experiment_ID'].isin(test_exp)]['Experiment_ID']

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

Training experiments: 14
Validation experiments: 3
Test experiments: 3

X_train shape: (127979, 4)
X_val shape: (5452, 4)
X_test shape: (26185, 4)


### StandardScaler

In [17]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Define the columns to be scaled
features_to_scale = ['Duration', 'Force', 'Elongation', 'Temperature']

# Fit the scaler ONLY on the training data
scaler.fit(X_train[features_to_scale])

# Transform the training, validation, and test data
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[features_to_scale] = scaler.transform(X_train[features_to_scale])
X_val_scaled[features_to_scale] = scaler.transform(X_val[features_to_scale])
X_test_scaled[features_to_scale] = scaler.transform(X_test[features_to_scale])

print("Scaled Training Data Head:")
display(X_train_scaled.head())

print("\nScaled Validation Data Head:")
display(X_val_scaled.head())

print("\nScaled Test Data Head:")
display(X_test_scaled.head())

print("\nMean of scaled training features (should be close to 0):")
print(X_train_scaled[features_to_scale].mean())

print("\nStandard deviation of scaled training features (should be close to 1):")
print(X_train_scaled[features_to_scale].std())

Scaled Training Data Head:


,Duration,Force,Elongation,Temperature
921,-0.986423,-2.259527,-1.825233,-14.801420
922,-0.986364,-2.259527,-1.825233,-14.801420
923,-0.986268,-2.259527,-1.825233,-14.696096
924,-0.986240,-2.259527,-1.825233,-14.590771
925,-0.986212,-2.259527,-1.825233,-14.485447



Scaled Validation Data Head:


,Duration,Force,Elongation,Temperature
423,-0.986423,-2.259527,-1.825233,-14.906745
424,-0.986420,-2.259527,-1.825233,-14.906745
425,-0.986279,-2.259527,-1.825233,-14.801420
426,-0.986244,-2.259527,-1.825233,-14.696096
427,-0.986201,-2.259527,-1.825233,-14.590771



Scaled Test Data Head:


,Duration,Force,Elongation,Temperature
0,-0.986423,-2.259527,-1.825233,-13.590190
1,-0.986398,-2.259527,-1.825233,-13.432203
2,-0.986381,-2.259527,-1.825233,-13.326879
3,-0.986366,-2.259527,-1.825233,-13.221554
4,-0.986230,-2.259527,-1.825233,-13.168892



Mean of scaled training features (should be close to 0):
Duration      -1.279187e-16
Force         -1.989846e-16
Elongation     2.203044e-16
Temperature    5.427216e-15
dtype: float64

Standard deviation of scaled training features (should be close to 1):
Duration       1.000004
Force          1.000004
Elongation     1.000004
Temperature    1.000004
dtype: float64


In [18]:
## Prepare DataFrames for Sequence Creation

assert len(X_train_scaled) == len(experiment_ids_train), \
    "Training features and experiment IDs have different lengths."

assert len(X_val_scaled) == len(experiment_ids_val), \
    "Validation features and experiment IDs have different lengths."

assert len(X_test_scaled) == len(experiment_ids_test), \
    "Test features and experiment IDs have different lengths."


# Reset indices so that feature rows and experiment IDs align correctly
train_df_sequences = X_train_scaled.reset_index(drop=True).copy()
val_df_sequences = X_val_scaled.reset_index(drop=True).copy()
test_df_sequences = X_test_scaled.reset_index(drop=True).copy()

experiment_ids_train = experiment_ids_train.reset_index(drop=True)
experiment_ids_val = experiment_ids_val.reset_index(drop=True)
experiment_ids_test = experiment_ids_test.reset_index(drop=True)


# Add Experiment_ID back to each scaled DataFrame
train_df_sequences["Experiment_ID"] = experiment_ids_train
val_df_sequences["Experiment_ID"] = experiment_ids_val
test_df_sequences["Experiment_ID"] = experiment_ids_test


# Display a sample
print("Head of train_df_sequences:")
display(train_df_sequences.head())


# Print dataset shapes
print("\nShapes of DataFrames prepared for sequence creation:")
print(f"Train: {train_df_sequences.shape}")
print(f"Validation: {val_df_sequences.shape}")
print(f"Test: {test_df_sequences.shape}")


# Check the number of unique experiments in each split
print("\nNumber of experiments in each split:")
print(f"Train: {train_df_sequences['Experiment_ID'].nunique()}")
print(f"Validation: {val_df_sequences['Experiment_ID'].nunique()}")
print(f"Test: {test_df_sequences['Experiment_ID'].nunique()}")


# Check for missing values
print("\nMissing values in each split:")
print("Train:")
print(train_df_sequences.isna().sum())

print("\nValidation:")
print(val_df_sequences.isna().sum())

print("\nTest:")
print(test_df_sequences.isna().sum())


# Confirm that experiments do not overlap across splits
train_ids = set(train_df_sequences["Experiment_ID"].unique())
val_ids = set(val_df_sequences["Experiment_ID"].unique())
test_ids = set(test_df_sequences["Experiment_ID"].unique())

assert train_ids.isdisjoint(val_ids), \
    "Data leakage detected: train and validation experiments overlap."

assert train_ids.isdisjoint(test_ids), \
    "Data leakage detected: train and test experiments overlap."

assert val_ids.isdisjoint(test_ids), \
    "Data leakage detected: validation and test experiments overlap."

print("\nNo experiment overlap detected between train, validation, and test sets.")

Head of train_df_sequences:


,Duration,Force,Elongation,Temperature,Experiment_ID
0,-0.986423,-2.259527,-1.825233,-14.801420,1511008DXMS
1,-0.986364,-2.259527,-1.825233,-14.801420,1511008DXMS
2,-0.986268,-2.259527,-1.825233,-14.696096,1511008DXMS
3,-0.986240,-2.259527,-1.825233,-14.590771,1511008DXMS
4,-0.986212,-2.259527,-1.825233,-14.485447,1511008DXMS



Shapes of DataFrames prepared for sequence creation:
Train: (127979, 5)
Validation: (5452, 5)
Test: (26185, 5)

Number of experiments in each split:
Train: 14
Validation: 3
Test: 3

Missing values in each split:
Train:
Duration         0
Force            0
Elongation       0
Temperature      0
Experiment_ID    0
dtype: int64

Validation:
Duration         0
Force            0
Elongation       0
Temperature      0
Experiment_ID    0
dtype: int64

Test:
Duration         0
Force            0
Elongation       0
Temperature      0
Experiment_ID    0
dtype: int64

No experiment overlap detected between train, validation, and test sets.


In [19]:
import numpy as np
import torch

sequence_length = 10
forecast_horizon = 1

input_features = [
    "Duration",
    "Force",
    "Elongation",
    "Temperature"
]

target_feature = "Elongation"


def create_sequences(
    df,
    sequence_length,
    forecast_horizon,
    input_features,
    target_feature
):
    X = []
    y = []

    for exp_id in df["Experiment_ID"].unique():

        # Select one experiment and sort it chronologically
        exp_df = (
            df[df["Experiment_ID"] == exp_id]
            .sort_values("Duration")
            .reset_index(drop=True)
        )

        # Select model inputs and target
        feature_data = exp_df[input_features].to_numpy(dtype=np.float32)
        target_data = exp_df[target_feature].to_numpy(dtype=np.float32)

        required_rows = sequence_length + forecast_horizon

        # Skip experiments that are too short
        if len(exp_df) < required_rows:
            print(
                f"Skipping {exp_id}: only {len(exp_df)} rows, "
                f"but at least {required_rows} are required."
            )
            continue

        number_of_sequences = len(exp_df) - required_rows + 1

        for i in range(number_of_sequences):

            # Historical input window
            X_window = feature_data[
                i:i + sequence_length
            ]

            # Future elongation target
            y_window = target_data[
                i + sequence_length:
                i + sequence_length + forecast_horizon
            ]

            X.append(X_window)
            y.append(y_window)

    if len(X) == 0:
        return (
            torch.empty(
                (0, sequence_length, len(input_features)),
                dtype=torch.float32
            ),
            torch.empty(
                (0, forecast_horizon),
                dtype=torch.float32
            )
        )

    X = torch.tensor(np.asarray(X), dtype=torch.float32)
    y = torch.tensor(np.asarray(y), dtype=torch.float32)

    return X, y

In [20]:
X_train_seq, y_train_seq = create_sequences(
    train_df_sequences,
    sequence_length,
    forecast_horizon,
    input_features,
    target_feature
)

X_val_seq, y_val_seq = create_sequences(
    val_df_sequences,
    sequence_length,
    forecast_horizon,
    input_features,
    target_feature
)

X_test_seq, y_test_seq = create_sequences(
    test_df_sequences,
    sequence_length,
    forecast_horizon,
    input_features,
    target_feature
)